# Исследовательский анализ данных (EDA): исходы цирроза печени

Исследуем данные соревнования [Kaggle Playground Series S3E26](https://www.kaggle.com/competitions/playground-series-s3e26): структуру и качество таблиц, распределения признаков и целевой переменной, связи между ними и различия между обучающей и тестовой выборками. Анализ будет дополняться по мере работы над проектом. Исходные файлы: `train.csv`, `test.csv`, `sample_submission.csv`.

## 1. Цель проекта и показатели эффективности

**Цель:** по табличным признакам пациента с циррозом предсказать вероятности трёх значений `Status`:

- `C` — пациент жив на момент `N_Days` (наблюдение цензурировано);
- `CL` — пациент жив после трансплантации печени;
- `D` — пациент умер к моменту `N_Days`.

Это **многоклассовая классификация**. Для каждой строки теста нужны три вероятности: `Status_C`, `Status_CL`, `Status_D`.

**Основная метрика соревнования — multiclass log loss** (чем меньше, тем лучше). Она оценивает качество *вероятностей* и сильно штрафует уверенные ошибочные прогнозы. Формула: $-\frac{1}{N}\sum_i\sum_j y_{ij}\log(p_{ij})$. При локальной проверке будем использовать стратифицированную кросс-валидацию и считать средний log loss со стандартным отклонением между фолдами. Предсказания для каждого пациента должны суммироваться до 1.

**Дополнительные диагностические метрики:** accuracy, macro F1, recall по каждому классу и матрица ошибок. Они помогают понять, как модель работает с редким `CL`, но не заменяют основную метрику. Для начала полезно сравнить будущие модели с базовым прогнозом по частотам классов на обучающей части каждого фолда.

**Ограничение постановки:** `N_Days` отражает длительность наблюдения. Признак дан в тесте Kaggle, но для реального прогноза на момент первичного осмотра его доступность нужно отдельно обосновать. Данные синтетические; результаты соревнования нельзя напрямую трактовать как клиническую точность.

## 2. Загрузка данных и первичный осмотр

Загружаем исходные CSV без изменения значений. `id` — идентификатор строки; целевой признак `Status` есть только в `train`. Пути ищутся относительно расположения ноутбука или текущей рабочей папки.

In [1]:
from pathlib import Path
import pandas as pd

# Работает при запуске из корня проекта и из папки с самим ноутбуком.
base_dir = Path.cwd()
if not (base_dir / "train.csv").exists():
    raise FileNotFoundError("Не найден train.csv. Запустите ноутбук из папки проекта.")

train = pd.read_csv(base_dir / "train.csv")
test = pd.read_csv(base_dir / "test.csv")
submission_example = pd.read_csv(base_dir / "sample_submission.csv")

print(f"train: {train.shape[0]} строк × {train.shape[1]} столбцов")
print(f"test: {test.shape[0]} строк × {test.shape[1]} столбцов")
print(f"sample_submission: {submission_example.shape[0]} строк × {submission_example.shape[1]} столбца")

train: 7905 строк × 20 столбцов
test: 5271 строк × 19 столбцов
sample_submission: 5271 строк × 4 столбца


### Первые строки и типы признаков

Первые строки показывают формат данных; `info()` помогает проверить типы столбцов и количество непустых значений.

In [2]:
print(train.head(5).to_string())
train.info()

   id  N_Days             Drug    Age Sex Ascites Hepatomegaly Spiders Edema  Bilirubin  Cholesterol  Albumin  Copper  Alk_Phos    SGOT  Tryglicerides  Platelets  Prothrombin  Stage Status
0   0     999  D-penicillamine  21532   M       N            N       N     N        2.3        316.0     3.35   172.0    1601.0  179.80           63.0      394.0          9.7    3.0      D
1   1    2574          Placebo  19237   F       N            N       N     N        0.9        364.0     3.54    63.0    1440.0  134.85           88.0      361.0         11.0    3.0      C
2   2    3428          Placebo  13727   F       N            Y       Y     Y        3.3        299.0     3.55   131.0    1029.0  119.35           50.0      199.0         11.7    4.0      D
3   3    2576          Placebo  18460   F       N            N       N     N        0.6        256.0     3.50    58.0    1653.0   71.30           96.0      269.0         10.7    3.0      C
4   4     788          Placebo  16658   F       N      

### Описательная статистика (`describe`)

Для числовых признаков смотрим количество значений, среднее, стандартное отклонение, квартили и крайние значения. Для категориальных — количество уникальных значений, самый частый вариант и его частоту. `id` — технический идентификатор, поэтому исключаем его из статистики признаков.

In [3]:
numeric_features = train.select_dtypes(include="number").drop(columns="id")
print(numeric_features.describe().T.to_string())

                count          mean          std      min       25%       50%       75%       max
N_Days         7905.0   2030.173308  1094.233744    41.00   1230.00   1831.00   2689.00   4795.00
Age            7905.0  18373.146490  3679.958739  9598.00  15574.00  18713.00  20684.00  28650.00
Bilirubin      7905.0      2.594485     3.812960     0.30      0.70      1.10      3.00     28.00
Cholesterol    7905.0    350.561923   195.379344   120.00    248.00    298.00    390.00   1775.00
Albumin        7905.0      3.548323     0.346171     1.96      3.35      3.58      3.77      4.64
Copper         7905.0     83.902846    75.899266     4.00     39.00     63.00    102.00    588.00
Alk_Phos       7905.0   1816.745250  1903.750657   289.00    834.00   1181.00   1857.00  13862.40
SGOT           7905.0    114.604602    48.790945    26.35     75.95    108.50    137.95    457.25
Tryglicerides  7905.0    115.340164    52.530402    33.00     84.00    104.00    139.00    598.00
Platelets      7905.

In [4]:
categorical_features = train.select_dtypes(exclude="number")
print(categorical_features.describe().T.to_string())

             count unique      top  freq
Drug          7905      2  Placebo  4010
Sex           7905      2        F  7336
Ascites       7905      2        N  7525
Hepatomegaly  7905      2        Y  4042
Spiders       7905      2        N  5966
Edema         7905      3        N  7161
Status        7905      3        C  4965


### Структура train и test

Проверяем, что тест отличается от обучающей выборки только отсутствием целевой переменной, а идентификаторы уникальны.

In [5]:
feature_columns = [column for column in train.columns if column != "Status"]
print("Признаки train и столбцы test совпадают:", feature_columns == list(test.columns))
print("Уникальные id в train:", train["id"].is_unique)
print("Уникальные id в test:", test["id"].is_unique)
print("Пересечение id train/test:", len(set(train["id"]) & set(test["id"])))
print("Столбцы файла отправки:", submission_example.columns.tolist())

Признаки train и столбцы test совпадают: True
Уникальные id в train: True
Уникальные id в test: True
Пересечение id train/test: 0
Столбцы файла отправки: ['id', 'Status_C', 'Status_CL', 'Status_D']


### Распределение целевой переменной

Редкость одного из классов важна для стратификации и интерпретации дополнительных метрик.

In [6]:
target_distribution = pd.DataFrame({
    "Количество": train["Status"].value_counts(),
    "Доля, %": train["Status"].value_counts(normalize=True).mul(100).round(2),
}).sort_index()
print(target_distribution.to_string())

        Количество  Доля, %
Status                     
C             4965    62.81
CL             275     3.48
D             2665    33.71


## 3. Анализ и обработка пропусков

Сначала считаем настоящие `NaN`/`None`. Отдельно проверяем пустые и состоящие из пробелов строки в текстовых столбцах: pandas может не считать их пропусками. Проверки проводим на `train` и `test` по отдельности.

In [7]:
def missing_report(frame):
    # В одном отчёте объединяем стандартные пропуски и пустой текст.
    text_columns = frame.select_dtypes(include=["object", "string"]).columns
    missing_mask = frame.isna().copy()
    for column in text_columns:
        missing_mask[column] |= frame[column].astype("string").str.strip().eq("").fillna(False)
    report = pd.DataFrame({
        "Пропусков": missing_mask.sum(),
        "Доля, %": missing_mask.mean().mul(100).round(2),
    })
    return report.sort_values("Пропусков", ascending=False), int(missing_mask.any(axis=1).sum())

for name, frame in [("train", train), ("test", test)]:
    report, rows_with_missing = missing_report(frame)
    print(f"{name}: строк с хотя бы одним пропуском — {rows_with_missing}; всего пропусков — {report['Пропусков'].sum()}")
    print(report.to_string())

train: строк с хотя бы одним пропуском — 0; всего пропусков — 0.0
               Пропусков  Доля, %
id                   0.0      0.0
N_Days               0.0      0.0
Drug                 0.0      0.0
Age                  0.0      0.0
Sex                  0.0      0.0
Ascites              0.0      0.0
Hepatomegaly         0.0      0.0
Spiders              0.0      0.0
Edema                0.0      0.0
Bilirubin            0.0      0.0
Cholesterol          0.0      0.0
Albumin              0.0      0.0
Copper               0.0      0.0
Alk_Phos             0.0      0.0
SGOT                 0.0      0.0
Tryglicerides        0.0      0.0
Platelets            0.0      0.0
Prothrombin          0.0      0.0
Stage                0.0      0.0
Status               0.0      0.0
test: строк с хотя бы одним пропуском — 0; всего пропусков — 0.0
               Пропусков  Доля, %
id                   0.0      0.0
N_Days               0.0      0.0
Drug                 0.0      0.0
Age                

### Решение по обработке

В текущих файлах пропусков и пустых текстовых значений нет, поэтому удаление строк или заполнение значений **не требуется**. Исходные CSV сохраняем без изменений. Если пропуски появятся в новых данных, способ заполнения нужно выбирать по смыслу признака и обучать его только на тренировочной части каждого фолда, чтобы не допустить утечки информации.